In [1]:
%pip install chromadb sentence-transformers numpy

  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached pybase64-1.4.3-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (8.7 kB)
  Using cached uvicorn-0.49.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.42.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached tqdm-4.68.3-py3-none-any.whl.metadata (57 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb

# Load embedding model once
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create Chroma client and collection
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="candidates")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1125.49it/s]


In [7]:
# TODO: Replace this with the real output from Phase 1 (structured extraction).
# Expected format: a list of dicts, one per candidate, each with:
#   - "id"   : a unique string identifier for the candidate (e.g. candidate filename, index, or DB id)
#   - "text" : a single string summarizing the candidate's profile for embedding —
#              likely built by combining extracted fields from Phase 1
#              (e.g. hard_skills + experience_scope + soft_skill signals joined into one string)
#
# Example of what Phase 1's output needs to be converted INTO before reaching this point:
# candidates = [
#     {"id": "cand_1", "text": "Python developer, 3 years experience, FastAPI, PostgreSQL"},
#     {"id": "cand_2", "text": "Frontend engineer, React, TypeScript, 2 years experience"},
#     ...
# ]

candidates = []  # <-- populate this from Phase 1's structured extraction output / the real dataset

# TODO: If the real dataset is structured (e.g. JSON with separate fields per candidate
# instead of one flat string), build the "text" field here by concatenating the relevant
# fields, e.g.:
# candidates = [
#     {"id": row["candidate_id"], "text": f"{row['hard_skills']} {row['experience_scope']} {row['soft_skills']}"}
#     for row in phase1_output
# ]

candidate_texts = [c["text"] for c in candidates]
candidate_ids = [c["id"] for c in candidates]

# Guard so this doesn't silently fail / crash confusingly if candidates is still empty
if not candidates:
    print("Warning: 'candidates' is empty — load the real dataset before running this cell.")
else:
    candidate_embeddings = model.encode(candidate_texts)

    collection.add(
        embeddings=candidate_embeddings,
        documents=candidate_texts,
        ids=candidate_ids
    )

    print(f"Added {collection.count()} candidates to the collection")

In [3]:
# Delete the old collection so we don't have duplicate/stale entries
chroma_client.delete_collection(name="candidates")
collection = chroma_client.get_or_create_collection(name="candidates")

candidates = [
    {"id": "cand_1", "text": "Python developer with 3 years experience in backend systems, FastAPI, PostgreSQL"},
    {"id": "cand_2", "text": "Frontend engineer skilled in React, TypeScript, 2 years experience"},
    {"id": "cand_3", "text": "Machine learning engineer, trained neural networks, 4 years experience with PyTorch"},
    {"id": "cand_4", "text": "Backend developer, Node.js and Express, deployed APIs to production"},
]

# Generate embeddings ourselves, explicitly
candidate_texts = [c["text"] for c in candidates]
candidate_embeddings = model.encode(candidate_texts)

print(f"Generated {len(candidate_embeddings)} embeddings")
print(f"Each embedding has {len(candidate_embeddings[0])} numbers")
print(f"First few numbers of cand_1's embedding: {candidate_embeddings[0][:5]}")

Generated 4 embeddings
Each embedding has 384 numbers
First few numbers of cand_1's embedding: [-0.02570286 -0.04281052 -0.02281692  0.05771822 -0.08489701]


In [8]:
def get_top_candidates(job_description: str, n: int = 50):
    """
    Takes a job description, returns the top N candidates
    with a normalized semantic_score between 0.0 and 1.0
    (1.0 = perfect match, 0.0 = unrelated).
    """
    jd_embedding = model.encode([job_description])

    results = collection.query(
        query_embeddings=jd_embedding,
        n_results=n
    )

    ids = results["ids"][0]
    documents = results["documents"][0]
    distances = results["distances"][0]

    # Convert distance -> similarity score (0 to 1, higher = better)
    # Chroma's default distance can exceed 1.0, so we normalize against the max seen
    max_distance = max(distances) if distances else 1.0
    output = []
    for cand_id, doc, dist in zip(ids, documents, distances):
        semantic_score = 1 - (dist / max_distance) if max_distance > 0 else 1.0
        output.append({
            "id": cand_id,
            "text": doc,
            "semantic_score": round(semantic_score, 4)
        })

    return output

In [10]:
# TODO: Replace this with the real job description for the role being matched.
# This should be the raw or lightly cleaned JD text — same kind of text that
# Phase 1's extraction step parses into structured fields, but here we just need
# the plain string for embedding.
#
# Example of what it looked like during testing:
# job_description = "Looking for a backend developer experienced with Python and APIs"

job_description = ""  # <-- populate this with the real JD text

if not job_description:
    print("Warning: 'job_description' is empty — set the real JD text before running this cell.")
elif not candidates:
    print("Warning: 'candidates' is empty — load the real dataset in Block 2 before running this cell.")
else:
    top_candidates = get_top_candidates(job_description, n=50)

    for c in top_candidates:
        print(f"{c['id']}: {c['semantic_score']} — {c['text'][:60]}")